## Notebook33

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
it_city = pl.read_csv(ub + "data/it_cities.csv")
prov = funs.read_file(ub + "data/it_province.geojson")

### Questions

This notebook brings back both Italy tables from the last two notebooks and
puts them together. `it_city` is the table of 388 cities, with `city_name`,
`lon`, `lat`, and `pop`; `prov` is the GeoJSON of Italy's 107 provinces, with
a `province` name and a polygon `geometry`. A row of `it_city` is one city; a
row of `prov` is one whole province boundary. The two are not linked by any
shared key column this time: there is no province name on the city table.
What links them is geography: each city sits inside exactly one province, and
the point of a spatial join is to recover that relationship from the
geometries alone.

Spatial operations only work when both tables are in the same coordinate
system, so the first question projects both into EPSG:7791 and saves them
back. After that, the join and geometry-building steps get reassigned; the
plain lookups and counts should just be printed.

1. Build a point `geometry` for `it_city` with `it_city.geo.points_from_xy(
"lon", "lat")`, then project it into EPSG:7791 and save it back. Project
`prov` into EPSG:7791 as well and save that too. Print both `_crs` columns to
confirm they now agree.

In [ ]:
it_city = it_city.geo.points_from_xy("lon", "lat").geo.to_crs(7791)
prov = prov.geo.to_crs(7791)

In [ ]:
it_city.select(c._crs.first())

In [ ]:
prov.select(c._crs.first())

**Answer**:


2. Join the cities into the provinces with `it_city.geo.sjoin(prov)` and print
the result. Each city point gets matched to the province polygon that contains
it. Look at the columns: what does a row mean now, and what are `index_right`
and the `geometry` column showing?

In [ ]:
it_city.geo.sjoin(prov)

**Answer**:


3. A spatial join is an inner join by default, so a city that fell outside
every province polygon would be dropped with no error. Guard against that:
compare the row count of the join to the row count of `it_city`. Did every
city match?

In [ ]:
it_city.geo.sjoin(prov).select(pl.len())

In [ ]:
it_city.select(pl.len())

**Answer**:


4. Reverse the join with `prov.geo.sjoin(it_city)`, so the province polygons
are kept instead. This gives one row per city again, but now the province
geometry is repeated for each city inside it. Group by `province` and count
the cities per province, then print the five provinces with the most.

In [ ]:
(
    prov
    .geo.sjoin(it_city)
    .group_by(c.province)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
    .head(5)
)

**Answer**:


5. Use the join to answer a question neither table can answer alone: how much
of each province's population lives in these cities? Join the cities into the
provinces, group by `province`, sum `pop`, and print the six provinces with
the largest urban population.

In [ ]:
(
    it_city
    .geo.sjoin(prov)
    .group_by(c.province)
    .agg(urban_pop = c.pop.sum())
    .sort(c.urban_pop, descending=True)
    .head(6)
)

**Answer**:


6. The city table has repeated names. Filter the join to the rows where
`city_name` is `"Massa"` and print `city_name`, `pop`, and `province`. Two
things are worth noticing: which provinces the three Massas land in, and what
their `pop` values are.

In [ ]:
(
    it_city
    .geo.sjoin(prov)
    .filter(c.city_name == "Massa")
    .select(c.city_name, c.pop, c.province)
)

**Answer**:


7. Provinces can be joined to *themselves*. Use `prov.geo.sjoin(prov,
predicate="touches")` to find every pair of provinces that share a border, and
print how many rows come back.

In [ ]:
prov.geo.sjoin(prov, predicate="touches").select(pl.len())

**Answer**:


8. Pull out one province's neighbors. Filter the `touches` self-join to the
rows where `province_left` is `"Milano"` and print the `province_right`
values. Do these match what you would expect from a map of Lombardy?

In [ ]:
(
    prov
    .geo.sjoin(prov, predicate="touches")
    .filter(c.province_left == "Milano")
    .select(c.province_right)
    .sort(c.province_right)
)

**Answer**:


9. Not every province has that many neighbors. Count neighbors per province
from the `touches` join (group by `province_left`, count), sort ascending, and
print the five with the fewest. What do the provinces at the bottom have in
common?

In [ ]:
(
    prov
    .geo.sjoin(prov, predicate="touches")
    .group_by(c.province_left)
    .agg(n = pl.len())
    .sort(c.n)
    .head(5)
)

**Answer**:


10. Now switch from overlap to proximity. Use `it_city.geo.sjoin_nearest(
it_city, exclusive=True, distance_col="distance")` to find each city's nearest
other city. Sort by `distance` descending and print the six most isolated
cities with their distance in kilometers (`c.distance / 1000`).

In [ ]:
(
    it_city
    .geo.sjoin_nearest(it_city, exclusive=True, distance_col="distance")
    .sort(c.distance, descending=True)
    .select(c.city_name_left, c.city_name_right, (c.distance / 1000).round(1).alias("km"))
    .head(6)
)

**Answer**:


11. The distances in question 10 are only meaningful because the table was
projected. Repeat the nearest-neighbor join on an *unprojected* copy of the
cities and look at what happens to the ranking. Load the cities fresh, build
points but leave them in EPSG:4326, run the same join, and print the four most
isolated by `distance`. Does the top of the list still agree with question 10?

In [ ]:
it_deg = (
    pl.read_csv(ub + "data/it_cities.csv")
    .geo.points_from_xy("lon", "lat")
)

In [ ]:
(
    it_deg
    .geo.sjoin_nearest(it_deg, exclusive=True, distance_col="distance")
    .sort(c.distance, descending=True)
    .select(c.city_name_left, c.city_name_right, c.distance.round(4).alias("deg"))
    .head(4)
)

**Answer**:
